# 🚧 Road Damage Detection: Custom YOLO11/YOLOv8 Training on Google Colab

This notebook provides a complete, optimized pipeline to train a custom YOLO model on the Road Damage Detection dataset. It includes:
1. Mounting Google Drive to permanently save model weights (`best.pt` / `last.pt`).
2. Installing required dependencies.
3. Downloading the dataset from Roboflow.
4. Dynamically correcting the `data.yaml` dataset paths.
5. Training an optimized **YOLO11s** model with advanced hyperparameters (AdamW, vertical flips, learning rate adjustments, and late-epoch mosaic disabling) to boost accuracy for small cracks/potholes.
6. Running validation and test set evaluations.
7. Executing inference on test images and saving the annotated outputs.

### Step 1: Mount Google Drive & Install Libraries
Mounting Google Drive ensures that if your Colab session times out or disconnects, your training runs, logs, and checkpoints (`best.pt`) are safely persisted on your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install Ultralytics and Roboflow libraries
!pip install ultralytics roboflow

### Step 2: Import Dependencies and Verify GPU
Verify that Colab is running with GPU acceleration (preferably Tesla T4, L4, or A100).

In [ ]:
import os
import yaml
import glob
import random
import cv2
from PIL import Image
from ultralytics import YOLO
from roboflow import Roboflow

# Check CUDA GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

### Step 3: Download Dataset from Roboflow
Retrieve the dataset programmatically. Enter your Roboflow Private API Key below. 
You can get your key by going to your Roboflow Account Settings -> Workspaces -> Copy Private API Key.

In [ ]:
# Paste your Roboflow API key below
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY_HERE"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("yolo-sfvlm").project("road-damages-detection-9gt0k")
version = project.version(1)
dataset = version.download("yolov8")

### Step 4: Fix `data.yaml` Directory Paths
Roboflow sometimes outputs absolute local paths in `data.yaml` which do not match the Colab directory structure. 
Running this cell ensures all train/val/test image folder paths are correctly resolved relative to the downloaded directory.

In [ ]:
dataset_location = dataset.location
yaml_path = os.path.join(dataset_location, "data.yaml")

# Read data.yaml
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Update path fields to reflect absolute Colab paths
data['path'] = dataset_location
data['train'] = "train/images"
data['val'] = "valid/images"
data['test'] = "test/images"

# Save updated data.yaml
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("--- Updated data.yaml configuration ---")
print(yaml.dump(data))

### Step 5: Train Optimized Custom YOLO11s Model
To solve the poor validation results (mAP@50 ~ 47%), we apply the following optimizations:
1. **Model Size**: We upgrade from Nano (`yolo11n.pt`) to Small (`yolo11s.pt`) which has higher capacity to learn complex textures.
2. **Optimizer**: We use `AdamW` instead of standard `SGD` with a smaller, stable learning rate (`lr0=0.001`) to prevent weight oscillations.
3. **Data Augmentations for Cracks**:
   * `flipud=0.5` (enable vertical flip) and `fliplr=0.5` (horizontal flip) since cracks can run vertically or horizontally on roads.
   * `close_mosaic=10` (turns off mosaic augmentation for the last 10 epochs). This allows the model to stabilize and refine bounding box coordinates for small objects before training ends.


In [ ]:
# Load standard pre-trained YOLO11s weights as base
model = YOLO("yolo11s.pt")

# Path in Google Drive to store runs permanently
drive_project_dir = "/content/drive/MyDrive/Road_Damage_Detection"
os.makedirs(drive_project_dir, exist_ok=True)

# Start training
results = model.train(
    data=yaml_path,
    epochs=80,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    patience=20,             # Early stopping threshold
    close_mosaic=10,         # Disable mosaic for last 10 epochs for fine-tuning
    flipud=0.5,              # Vertical flips
    fliplr=0.5,              # Horizontal flips
    mosaic=1.0,              # Mosaic data mix
    mixup=0.15,              # Blend images
    copy_paste=0.1,
    project=drive_project_dir,
    name="yolo11s_optimized",
    val=True,
    device=0                 # Use GPU index 0
)

### Step 6: Evaluate Model on Test Dataset
Evaluate your best model checkpoint on the unused **Test Set** split to see its true, unbiased generalize accuracy.

In [ ]:
# Load the best checkpoint from Google Drive
best_model_path = os.path.join(drive_project_dir, "yolo11s_optimized", "weights", "best.pt")
best_model = YOLO(best_model_path)

# Evaluate on test split
metrics = best_model.val(data=yaml_path, split="test")

print("--- Test Set Performance Metrics ---")
print(f"mAP@50: {metrics.results_dict['metrics/mAP50(B)'] * 100:.2f}%")
print(f"mAP@50-95: {metrics.results_dict['metrics/mAP50-95(B)'] * 100:.2f}%")
print(f"Precision: {metrics.results_dict['metrics/precision(B)'] * 100:.2f}%")
print(f"Recall: {metrics.results_dict['metrics/recall(B)'] * 100:.2f}%")

### Step 7: Run Inference on 10 Random Test Images
Run predictions on 10 test images, display the bounding boxes, and save them to Google Drive for submission.

In [ ]:
test_images_dir = os.path.join(dataset_location, "test", "images")
test_images = glob.glob(os.path.join(test_images_dir, "*.jpg")) + glob.glob(os.path.join(test_images_dir, "*.png"))

# Select 10 random images
selected_test_imgs = random.sample(test_images, min(10, len(test_images)))

# Output directory in Google Drive to hold results
test_results_dir = os.path.join(drive_project_dir, "test_predictions_output")
os.makedirs(test_results_dir, exist_ok=True)

for i, img_path in enumerate(selected_test_imgs):
    # Predict
    res = best_model.predict(img_path, conf=0.25, verbose=False)[0]
    
   # Get annotated image
    annotated_bgr = res.plot()
    
    # Save to Drive
    out_path = os.path.join(test_results_dir, f"prediction_{i+1:02d}.jpg")
    cv2.imwrite(out_path, annotated_bgr)
    print(f"Processed image {i+1}/10: Saved annotated result to {out_path}")